# [P2] Q2 A + B: data audit and cleaning

**Owner:** Part 2  
**Temporary part notebook.** At integration it is merged into `Q2_laptop_analysis.ipynb` and this file is deleted.


## Before writing any code

Tasks: `t_q2a` (audit), `t_q2b` (cleaning and new variables, with validation).

Decisions to read on the team board first (board ids): `q_data q_dups q_issues` (A), `os_map st_type rare validate` (B), and the shared `st_extra`.

Do not write code for a task until its decisions are made. Mark your cells with the part tag so the merge is easy.


# Q2 A: Data audit

Before analysing the laptop data we investigate its structure and quality. This notebook section applies the team decisions below.

| Decision | What we chose |
|---|---|
| `q_data` | The dataset is a copy in `data/laptop_price_dataset.csv` (no space in the name). The raw file is never modified. |
| `q_dups` | Near-duplicate rows are kept and documented; at the end we check that the conclusions hold without them. |
| `q_issues` | Which problems we address and which we only document (see the register in section 6). |
| `q_audit` | The audit contains: a variables table, a statement of what a row is, data type checks, range and unit checks, and consistency checks between columns. |
| `g_lang`, `g_nb` | English, and every step is explained in a text cell before its code. |

## 0. Setup

We import the libraries and read the file with `load_raw()`. This function only reads: no column is changed, no row is dropped. Keeping the raw table untouched lets us always compare against the original.

In [ ]:
# HE: ייבוא ספריות. numpy היא ספרייה לחישובים על מספרים. pandas היא ספרייה לעבודה עם טבלאות.
# HE: מקובל לקצר את השמות: np ו-pd.
import numpy as np
import pandas as pd

# HE: load_raw היא הפונקציה שכתבנו בקובץ q2_clean.py. היא רק קוראת את קובץ הנתונים.
from q2_clean import load_raw

# HE: שתי השורות הבאות משנות רק איך הטבלאות מוצגות על המסך (רוחב ומספר עמודות מקסימלי).
# HE: הן לא משנות את הנתונים עצמם.
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

# HE: קוראים את הנתונים למשתנה raw (גולמי). לעולם לא משנים את raw עצמו: כל ניקוי יעשה על עותק.
raw = load_raw()

# HE: raw.shape נותן זוג מספרים: (כמה שורות, כמה עמודות).
print("shape (rows, columns):", raw.shape)

# HE: raw.head(3) מציג את שלוש השורות הראשונות, כדי לראות איך הנתונים נראים.
# HE: בתא במחברת, השורה האחרונה מוצגת אוטומטית במסך.
raw.head(3)

## 1. What does one row represent?

Decision (`q_audit`, row definition): we state in one sentence what a row is, because it determines what counts as one observation.

**Each row is a laptop listing: one configuration (brand, model, screen, CPU, RAM, storage, GPU, operating system, weight) together with one price in euros.** A row is not necessarily a unique model: the same configuration can appear several times with different prices. We check this in section 4.

## 2. Variables table

Decision (`q_audit`, variables table): one row per column with its pandas data type, its kind, the number of different values, the number of missing values, and an example. The *kind* is assigned by us from the meaning of the column, and `assert` makes sure that no column is left unclassified.

- **numeric**: values we can compute with (price, weight, RAM...).
- **categorical**: labels or types, with no arithmetic meaning (company, operating system...).
- **structured text**: text that contains several pieces of information in one string and has to be parsed before use (for example `Memory` holds both capacity and storage type).

In [ ]:
# HE: אנחנו מסווגים כל עמודה לפי המשמעות שלה, ולא לפי מה שפייתון חושב עליה.
# HE: NUMERIC: מספרים שאפשר לחשב איתם. CATEGORICAL: תוויות (חברה, מערכת הפעלה).
# HE: STRUCTURED_TEXT: טקסט שמכיל כמה פרטים במחרוזת אחת (למשל Memory: "256GB SSD + 1TB HDD").
NUMERIC = ["Inches", "CPU_Frequency (GHz)", "RAM (GB)", "Weight (kg)", "Price (Euro)"]
CATEGORICAL = ["Company", "TypeName", "CPU_Company", "GPU_Company", "OpSys"]
STRUCTURED_TEXT = ["Product", "ScreenResolution", "CPU_Type", "Memory", "GPU_Type"]

# HE: kind הוא מילון (dictionary): לכל שם עמודה יש סוג. הכתיב {c: "numeric" for c in NUMERIC}
# HE: אומר "לכל עמודה c ברשימה NUMERIC, שייך לה את הערך numeric". update מוסיף למילון עוד זוגות.
kind = {c: "numeric" for c in NUMERIC}
kind.update({c: "categorical" for c in CATEGORICAL})
kind.update({c: "structured text" for c in STRUCTURED_TEXT})

# HE: assert הוא בדיקה: אם התנאי לא נכון, התוכנית נעצרת עם ההודעה שבסוף.
# HE: כאן בודקים שכל עמודה בטבלה סווגה, ושלא כתבנו שם עמודה לא קיים. כך לא נשכח עמודה בשקט.
assert set(kind) == set(raw.columns), "every column must be classified exactly once"

# HE: בונים טבלת סיכום עם שורה לכל עמודה:
# HE: dtype = סוג הנתונים לפי פייתון. n_unique = כמה ערכים שונים יש בעמודה.
# HE: n_missing = כמה תאים ריקים (isna מזהה ריקים, sum סופר אותם).
# HE: example = ערך לדוגמה מהשורה הראשונה. reindex(raw.columns) שומר על סדר העמודות של הקובץ.
variables = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "kind": pd.Series(kind).reindex(raw.columns),
    "n_unique": raw.nunique(),
    "n_missing": raw.isna().sum(),
    "example": raw.iloc[0],
})
variables

**What the table shows**

- The table has 1,275 rows and 15 columns: 5 numeric, 5 categorical and 5 structured-text columns.
- **No column has missing values** (`n_missing` is 0 everywhere).
- The numeric columns already have numeric types (`float64` or `int64`), so they can be used in calculations as they are. The text columns have the pandas text type (`str`).
- Several text columns have many different values: `Product` (618), `GPU_Type` (106), `CPU_Type` (93), `ScreenResolution` (40) and `Memory` (39). They mix several pieces of information in one string, and `Memory` in particular has to be parsed to get the storage capacity and the storage type (section 2 of the cleaning notebook, task Q2 B).
- `Price (Euro)` has 791 different values among 1,275 rows: prices repeat, which is expected when the same configuration is listed more than once. We look at this in section 4.

## 3. Data types: are they appropriate for the analysis?

Decision (`q_audit`, data types): we check that the columns we want to compute with are really numeric, and we list the text columns that must be converted before use, with evidence from the data.

In [ ]:
# HE: בודקים שהעמודות המספריות באמת בסוג מספרי. אם עמודה כזו הייתה נקראת כטקסט, לא היה אפשר לחשב איתה.
# HE: is_numeric_dtype מחזירה True אם סוג העמודה מספרי. המילון numeric_ok שומר תשובה לכל עמודה.
numeric_ok = {c: pd.api.types.is_numeric_dtype(raw[c]) for c in NUMERIC}
print(numeric_ok)
assert all(numeric_ok.values()), "every numeric column must have a numeric dtype"

# HE: עכשיו אוספים ראיות לכך שעמודות הטקסט מערבבות כמה פרטים במחרוזת אחת.
# HE: str.contains בודקת לכל שורה אם הטקסט מכיל תבנית, ו-sum סופר בכמה שורות זה נכון.
# HE: regex=False אומר "חפש את הטקסט כמו שהוא" (חשוב לסימן + שיש לו משמעות מיוחדת בביטוי רגולרי).
mem = raw["Memory"]
res = raw["ScreenResolution"]
evidence = pd.Series({
    "Memory: two drives (contains '+')": int(mem.str.contains("+", regex=False).sum()),
    "Memory: capacity in TB": int(mem.str.contains("TB", regex=False).sum()),
    "Memory: contains a double space": int(mem.str.contains("  ", regex=False).sum()),
    "Memory: Flash Storage": int(mem.str.contains("Flash", regex=False).sum()),
    "Memory: Hybrid": int(mem.str.contains("Hybrid", regex=False).sum()),
    "ScreenResolution: has a WxH resolution": int(res.str.contains(r"\d+x\d+").sum()),
    "ScreenResolution: Touchscreen": int(res.str.contains("Touchscreen", regex=False).sum()),
    "ScreenResolution: IPS panel": int(res.str.contains("IPS", regex=False).sum()),
})
evidence

Columns to convert, with the new variables they will produce (decision `st_extra` and the cleaning decisions `os_map` and `st_type`; the code is written in task Q2 B):

In [ ]:
# HE: זו רשימה ידנית (כתבנו אותה בעצמנו, אין כאן חישוב): איזו עמודה גולמית צריך לפרק,
# HE: למה, ואילו עמודות חדשות היא תיתן. היא מסכמת את ההחלטות שקיבלנו בלוח.
# HE: pd.DataFrame(רשימה של מילונים) בונה טבלה, כשכל מילון הוא שורה.
to_convert = pd.DataFrame([
    {"raw column": "Memory", "problem": "capacity and storage type in one string, 1-2 drives, GB and TB",
     "new variables": "Storage_type, Storage_GB_total, has_SSD"},
    {"raw column": "ScreenResolution", "problem": "resolution and panel features in one string",
     "new variables": "Screen_pixels, Touchscreen"},
    {"raw column": "CPU_Type", "problem": "processor level and model together",
     "new variables": "CPU_tier (Core i3 / i5 / i7 / other)"},
    {"raw column": "OpSys", "problem": "same system under two names (macOS and Mac OS X)",
     "new variables": "OpSys_clean, OS_family"},
    {"raw column": "GPU_Company", "problem": "needed as a yes/no for a dedicated graphics card",
     "new variables": "Dedicated_GPU"},
    {"raw column": "Price (Euro)", "problem": "right-skewed, multiplicative effects",
     "new variables": "log_price"},
])
to_convert

**What section 3 shows**

- All five numeric columns already have numeric types, so no conversion is needed for them.
- `Memory` mixes capacity and storage type: 208 laptops have two drives, 455 use TB units, 71 use Flash Storage and 10 use Hybrid. The same 208 strings contain a double space, so the parser in Q2 B must handle double spaces and both GB and TB.
- `ScreenResolution` contains a `WxH` resolution in all 1,275 rows, so the number of pixels can be extracted for every laptop. It also holds panel features: 188 touchscreens and 357 IPS panels.

## 4. Missing values and duplicates

Decisions: `q_audit` (missing values) and `q_dups` (near-duplicate rows are kept and documented). We look for missing values in three ways: real empty cells, "hidden" missing values (blank text, stray spaces) and impossible numbers (zero or negative).

In [ ]:
# HE: חסרים "אמיתיים": תאים ריקים. isna() מזהה ריקים, sum() סופר אותם, ושוב sum() מסכם את כל הטבלה.
print("empty cells in the whole table:", int(raw.isna().sum().sum()))

# HE: חסרים "מוסתרים": טקסט ריק או רווחים מיותרים שלא נראים לעין.
# HE: str.strip() מוחקת רווחים בתחילת ובסוף המחרוזת. אם הטקסט שונה אחרי strip, היו שם רווחים מיותרים.
text_cols = CATEGORICAL + STRUCTURED_TEXT
hidden = pd.DataFrame({
    "blank_text": {c: int((raw[c].str.strip() == "").sum()) for c in text_cols},
    "leading_or_trailing_space": {c: int((raw[c] != raw[c].str.strip()).sum()) for c in text_cols},
    "double_space_inside": {c: int(raw[c].str.contains("  ", regex=False).sum()) for c in text_cols},
})
print("text columns with any hidden problem:")
print(hidden[(hidden > 0).any(axis=1)])

# HE: מספרים בלתי אפשריים: משקל, מחיר וגודל מסך חייבים להיות חיוביים.
print("non-positive values per numeric column:")
print((raw[NUMERIC] <= 0).sum())

In [ ]:
# HE: כפילויות מדויקות: שורות זהות בכל העמודות. duplicated() מסמנת שורה שכבר הופיעה קודם.
print("exact duplicate rows:", int(raw.duplicated().sum()))

# HE: כמעט-כפילויות: שורות זהות בכל העמודות חוץ מהמחיר (אותה תצורה, מחיר שונה).
# HE: key_cols הן כל העמודות חוץ מהמחיר. subset=key_cols אומר "השווה רק לפי העמודות האלה".
# HE: keep=False מסמן את כל השורות בקבוצה (ולא רק את הכפולה), כדי שנראה את כולן.
# HE: near_dup_mask היא רשימת True/False לכל שורה. נשמור אותה: בסוף נחזור אליה לבדיקת רגישות.
key_cols = [c for c in raw.columns if c != "Price (Euro)"]
near_dup_mask = raw.duplicated(subset=key_cols, keep=False)
print("rows in near-duplicate groups:", int(near_dup_mask.sum()))

# HE: groupby מקבצת את השורות לפי התצורה, ו-agg מחשבת לכל קבוצה: כמה שורות, מחיר מינימלי ומקסימלי.
# HE: ratio = מחיר גבוה חלקי מחיר נמוך באותה תצורה. 1.10 אומר שההפרש הוא 10 אחוז.
dup_groups = (raw[near_dup_mask]
              .groupby(key_cols)["Price (Euro)"]
              .agg(rows="count", min_price="min", max_price="max")
              .reset_index())
dup_groups["ratio"] = (dup_groups["max_price"] / dup_groups["min_price"]).round(3)
print("near-duplicate groups:", len(dup_groups))
print("groups where the price differs:", int((dup_groups["ratio"] > 1).sum()))
dup_groups.sort_values("ratio", ascending=False)[
    ["Company", "Product", "RAM (GB)", "Memory", "rows", "min_price", "max_price", "ratio"]
].head(5)

**What section 4 shows**

- **Missing values:** there are no empty cells and no blank text. The only hidden problem is the 208 `Memory` strings with a double space (the two-drive entries, see above). There are no zero or negative numbers.
- **Exact duplicates:** none.
- **Near-duplicates:** 48 rows form 24 pairs that are identical in every column except the price, and in all 24 pairs the prices differ. The largest relative difference is a Lenovo Yoga Book (319 vs 549 euro, a ratio of 1.72). We do not know why the same configuration has different prices (different stores, dates or listing errors).
- **Decision `q_dups`:** we keep these rows and document them. They are not independent observations, so p-values computed on the full table are slightly optimistic. The mask `near_dup_mask` is kept so that, at the end of the analysis, we can check that the conclusions hold without them. This is also listed as a limitation of the report.